In [14]:
import random
import json
import os
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd


import os
from openai import AsyncOpenAI 
from agents import Agent, Runner 
from agents import set_default_openai_client
from agents import OpenAIResponsesModel 
from agents import set_tracing_disabled
set_tracing_disabled(True)#for jupyter, remove before moving to CLI
from pandas import read_csv

import numpy as np

### Playbook definition

In [15]:
class Playbook:
    def __init__(self, jsonl_path: str, docs_root: str = "docs/playbook"):
        self.jsonl_path = Path(jsonl_path)
        self.docs_root = Path(docs_root)

        # --- Load & parse JSONL --------------------------------------------
        self.chunks: List[Dict[str, Any]] = []

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue  # skip blank lines
                try:
                    chunk = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Malformed JSON on line {line_no} of {self.jsonl_path}"
                    ) from exc
                self.chunks.append(chunk)

        # Build quick look‑ups
        self.by_id: Dict[str, Dict[str, Any]] = {c["chunk_id"]: c for c in self.chunks}
        self.tag_index: Dict[str, List[str]] = {}
        for c in self.chunks:
            for tag in c["tags"]:
                self.tag_index.setdefault(tag, []).append(c["chunk_id"])

    # ------------------------------------------------------------
    #  API helpers
    # ------------------------------------------------------------
    def get_chunks_by_tags(self, tags: List[str]) -> List[Dict[str, Any]]:
        """Return the intersection of chunks that contain *all* supplied tags."""
        if not tags:
            return []

        # Start with the set for the first tag
        common = set(self.tag_index.get(tags[0], []))
        for t in tags[1:]:
            common &= set(self.tag_index.get(t, []))

        return [self.by_id[cid] for cid in common]

    def load_markdown_for_chunk(self, chunk: Dict[str, Any]) -> str:
        """
        The `doc_id` field points at the markdown file name *without* extension.
        e.g.  `doc_id="medical_necessity.md"` → load `docs/playbook/medical_necessity.md`
        """
        doc_file = self.docs_root / f"{chunk['doc_id']}"
        if not doc_file.exists():
            return f"[ERROR: missing {doc_file}]"

        return doc_file.read_text(encoding="utf-8")

    def load_markdown_for_tags(self, tags: List[str]) -> Dict[str, str]:
        """Return a mapping of doc_id → file contents for all chunks matching the tags."""
        chunks = self.get_chunks_by_tags(tags)
        return {c["doc_id"]: self.load_markdown_for_chunk(c) for c in chunks}


# claim_playbook = Playbook('data/playbook_chunks.jsonl')

In [16]:
# ───────────────────────────────────────────────────────────────────────
#  Imports
# ───────────────────────────────────────────────────────────────────────
from datetime import datetime

# Agents‑SDK core objects
from agents.agent import Agent
from agents import SQLiteSession
from agents import FunctionTool

# The Responses‑only model provider
from agents import OpenAIResponsesModel

In [18]:
# ------------------------------------------------------------
#  my_agent.py
# ------------------------------------------------------------
#  1️⃣  Dependencies ----------------------------------------------------
#      pip install openai-agents  # latest release (2026‑03‑20)
#
#  2️⃣  High‑level design ---------------------------------------------
#      • Uses the *Responses* API exclusively (no Chat Completions).
#      • Calls Ollama via its OpenAI‑compatible `/v1/responses` endpoint.
#      • Implements tools as **Agents‑SDK FunctionTools**.
#      • Stores multi‑turn conversation in a SQLite session store.
# ------------------------------------------------------------



# ───────────────────────────────────────────────────────────────────────
#  Agent wrapper class
# ───────────────────────────────────────────────────────────────────────
class OllamaResponsesAgent:
    """
    A skeleton Agent that satisfies the following constraints:

    • Uses the OpenAI Responses API exclusively (no chat completions).
    • Connects to an Ollama instance via its OpenAI compatibility layer
      (`/v1/responses`).
    • Exposes at least one FunctionTool (here: get_current_time).
    • Persists conversation state in a SQLite session store.
    """

    def __init__(
        self,
        *,
        ollama_base_url: str = "http://localhost:11434",
        model_name: str = "llama3.1",
        sqlite_uri: str = "sqlite:///erisa_agentic.db",
    ):
        """
        Initialise the agent, its memory store, and the Ollama Responses model.

        Parameters
        ----------
        ollama_base_url : str
            Base URL of the Ollama instance (must expose `/v1/responses`).
        model_name : str
            Name of the model to use inside Ollama.
        sqlite_uri : str
            URI for the SQLite session store.
        """
        # # 1️⃣  ── SQLite session store (multi‑turn persistence)
        self.session = SQLiteSession("conversation_123")

        client = AsyncOpenAI(
            api_key='None',
            base_url=ollama_base_url,  # key detail: route requests to Ollama (local or cloud)
        )
        
        set_default_openai_client(client)

        model = OpenAIResponsesModel(
            model='gpt-oss:20b',
            openai_client=client,
        )
        
        self.tools = [
            self._predict_denial_taxonomy_tool(),
            self._retrieve_playbook_tool(),
        ]

        self.agent = Agent(
            model=self.model,
            tools=self.tools,
            memory=self.session,            # multi‑turn memory
            instructions="""
                        You are a patient advocate. 
                        You will be given a claim with relevant information and your job is to suggest a recommendation for how to best proceed.
                        This recommendation can be one of the following options and nothing else: pursue, do_not_pursue, or needs_info. Always use
                        the retrieve_playbook tool to get more information and instructions specific to the type of claim and denial code.
                    
                        Afterwards, justify your recommendation.
                        """
        )
        self.playbook = Playbook('data/playbook_chunks.jsonl')
        self.tag_agent = Agent(
                name="Tag Assigning Agent",
                instructions="""
                You will be given an insurance claim and asked to assign any number of tags to it.
                If you find any of the following in the claim, it MUST be one of your tags: CO-16, CO-27, CO-29, CO-45, CO-50, CO-97
                Also assign any of the following tags if it appropriate: coding_bundling, eligibility, general, medical_necessity, missing_info, other, timely_filing, underpayment
                """,
                model=model,
            )
    def _predict_denial_taxonomy_tool(self) -> FunctionTool: #todo####################################################################################################
        """Return a FunctionTool that gives the current UTC time."""

        def predict_denial_taxonomy() -> str:
            """Predict denial for claim"""
            denial_reason = ['coding_bundling', 'eligibility', 'medical_necessity','missing_info', 'other', 'timely_filing', 'underpayment']
            return denial_reason[random.randint(0,len(denial_reason)-1)]

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="predict_denial_taxonomy",
            description="predict denial reason for a claim",
            function=predict_denial_taxonomy,
        )
    def _retrieve_playbook_tool(self) -> FunctionTool:
        """Get additional instructions and context from playbook using denial code and denial reason"""

        def retrieve_playbook() -> str:
            """Get additional instructions and context from playbook using denial code and denial reason"""
            # tags = await Runner.run(self.tag_agent, tag_prompt) 
            tags = Runner.run(self.tag_agent, tag_prompt) 
            return self.playbook.load_markdown_for_tags(list(tags))

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="retrieve_playbook",
            description="Get additional instructions and context from playbook using denial code and denial reason",
            function=retrieve_playbook,
            schema={
                    "type": "object",
                    "properties": {
                        "claim_information": {"type": "integer", "description": "any information about the claim"}, #specifying what claim info could help here
                    },
                    "required": ["claim_information"],
                },
        )

    # ------------------------------------------------------------------
    #  Public API: run a single turn
    # ------------------------------------------------------------------
    def run(self, prompt: str) -> str:
        """
        Send a prompt to the Agent and receive its response.

        The response may include tool calls; the Agent handles executing
        those tools automatically (via the FunctionTool infrastructure).

        Parameters
        ----------
        prompt : str
            The user message to send to the Agent.

        Returns
        -------
        str
            The raw text response from the Agent (after any tool calls).
        """
        # The Agent's `run` method uses the Responses API under the hood.
        result = self.agent.run(prompt,session=self.session)

        # `result` is typically a `Response` object – we extract the text.
        # Depending on the Agents‑SDK version you might need to adjust this.
        return result.text

    # ------------------------------------------------------------------
    #  Optional: Multi‑turn dialogue helper
    # ------------------------------------------------------------------
    def episode(self, messages: list[dict[str, str]]) -> str:
        """
        Send a list of user+assistant messages to the Agent and get the reply.

        Each dict should contain at least a `"role"` key with values `"user"` or `"assistant"`
        and a `"content"` key with the text.

        Parameters
        ----------
        messages : list[dict]
            Conversation history.

        Returns
        -------
        str
            The Agent's reply.
        """
        # Wrap the raw messages into the Agent‑SDK `Conversation` format if needed.
        # For brevity, we forward the list directly; adjust for your SDK version.
        result = self.agent.run(messages)
        return result.text


# ------------------------------------------------------------
#  Usage example (outside of the class definition)
# ------------------------------------------------------------
if __name__ == "__main__":
    agent = OllamaResponsesAgent()

    # # Single turn
    # print("👉 Prompt →")
    # print(agent.run("What is the current time?"))
    # print()

    # # Multi‑turn example
    # conversation = [
    #     {"role": "user", "content": "Hello!"},
    #     {"role": "assistant", "content": "Hi! How can I help?"},
    #     {"role": "user", "content": "Can you tell me the time?"},
    # ]

    # print("👉 Multi‑turn chat →")
    # print(agent.episode(conversation))

    claims = read_csv('data/claims.csv')
    for i,row in claims.iterrows():
        claim_dict = row.to_dict()
        result = agent.run(str(claim_dict))
        print('Claim ',i,'\n',result.final_output)
        if i <= 5:
            break
            

TypeError: __init__() got an unexpected keyword argument 'function'